In [ ]:
import pandas as pd
from plotnine import *
import numpy as np
from vpop_calibration.api import *

%load_ext autoreload
%autoreload 2

In [ ]:
"""
Friberg et al (2002) 'Model of Chemotherapy-Induced Myelosuppression with
Parameter Consistency Across Drugs', Journal of Clinical Oncology 20:4713-4721
"""

df = pd.read_csv("Simulated_WBC_pacl_ddmore_samePK_nlmixr.csv")

d = df[df.EVID == 10101].sort_values(["ID", "TIME"])
starts = d[d.RATE > 0].assign(k=lambda x: x.groupby("ID").cumcount() + 1)
stops = d[d.RATE < 0].assign(k=lambda x: x.groupby("ID").cumcount() + 1)
inf = starts.merge(
    stops[["ID", "k", "TIME"]], on=["ID", "k"], suffixes=("Start", "Stop")
)

wide = inf.pivot(index="ID", columns="k", values=["RATE", "TIMEStart", "TIMEStop"])
wide.columns = [f"{c}{k}" for c, k in wide.columns]
wide = (
    wide.rename(
        columns={
            "RATE1": "Rate1",
            "TIMEStop1": "Infusion1StopTime",
            "RATE2": "Rate2",
            "TIMEStart2": "Infusion2StartTime",
            "TIMEStop2": "Infusion2StopTime",
        }
    )
    .drop(columns="TIMEStart1")  # always 0, which the model assumes
    .fillna(0)  # single-infusion patients: rate 0
    .reset_index()
    .rename(columns={"ID": "id"})
)
obs_df = (
    df[df.EVID == 0][["ID", "DV", "V1I", "V2I", "CLI", "TIME"]]
    .rename(
        columns={
            "ID": "id",
            "DV": "value",
            "V1I": "V1",
            "V2I": "V2",
            "CLI": "CL",
            "TIME": "time",
        }
    )
    .astype({"V1": float, "V2": float, "CL": float})
)

obs_df = obs_df.merge(wide, on="id")
obs_df["value"] = np.log(obs_df["value"])
obs_df["output_name"] = "logWBC"
obs_df["time"] = obs_df["time"] * 3600  # convert from hours to seconds
obs_df["protocol_arm"] = "identity"
(
    ggplot(obs_df, aes(x="time / 3600", y="value", group="id"))
    + geom_line(alpha=0.6)
    + geom_point(size=1, alpha=0.8)
    + theme(legend_position="none")
    + labs(
        x="time (hour)",
        y="log(WBC [10^9/L])",
    )
)

In [ ]:
struct_model = StructuralJinko(
    JinkoModelBinding(
        path_to_model="CM_Friberg.json",
        path_to_solving_options="SolvingOptions_Friberg.json",
        inputs=[
            "CIRC0",
            "MTT",
            "SLOPU",
            "GAMMA",
            "CL",
            "V1",
            "V2",
            "Rate1",
            "Rate2",
            "Infusion1StopTime",
            "Infusion2StartTime",
            "Infusion2StopTime",
        ],
        outputs=["logWBC"],
    )
)

In [ ]:
prior_pdu = {
    "pdu": {
        "CIRC0": {"prior": 7.21, "prior_omega": 0.1},
        "MTT": {"prior": 124, "prior_omega": 0.03},
        "SLOPU": {"prior": 28.9, "prior_omega": 0.2},
    },
    "model_intrinsic": {
        "GAMMA": {"prior": 0.239},
    },
    "pdk": {
        "Rate1",
        "Rate2",
        "Infusion1StopTime",
        "Infusion2StartTime",
        "Infusion2StopTime",
        "CL",
        "V1",
        "V2",
    },
    "error_model": {
        "logWBC": {"error_type": "additive", "initial_variance": 0.5},
    },
}

config = Config(
    saem=SaemConfigDict(
        nb_iter_burnin=10,
        nb_iter_learning=90,
        nb_iter_smoothing=100,
        plot_frames=5,
    ),
    nlme=NlmeConfigDict(nb_chains=3),
)

nlme_model = NlmeModel(
    df=obs_df, input_params=prior_pdu, structural_model=struct_model, config=config
)
nlme_model.initial_estimates.plot_distribution()

In [ ]:
nlme_model.optimizer.run()

In [ ]:
nlme_model.diagnostics.compute_population_diagnostics(nb_samples=100)
nlme_model.plot.weighted_residuals("pwres")
nlme_model.plot.weighted_residuals("npde")

In [ ]:
nlme_model.plot.vpc()

In [ ]:
nlme_model.sample_conditional_distribution(100)
nlme_model.plot.map_estimates()
nlme_model.plot.map_estimates_gof()

In [ ]:
nlme_model.plot.weighted_residuals("iwres")

In [ ]:
nlme_model.plot.conditional_codistributions(figsize=[7, 7])

In [ ]:
df = nlme_model.diagnostics.report_shrinkage(nb_samples=100)
display(df.round(1))

In [ ]:
nlme_model.diagnostics.compute_log_likelihood_importance_sampling(
    nb_proposal_samples=100
)
print(
    f"Observed-data log-likelihood estimate, -2LL = {-2*nlme_model.diagnostics.importance_sampler.log_lik:.2f}"
)